# Step 1 — Data Pipeline, Drive Persistence, and Visual Verification

**Project:** Continual Self-Supervised Pretraining for Industrial Anomaly Localization

**Goal of this notebook:**
1. Mount Google Drive (persistence across Colab disconnects — non-negotiable)
2. Download MVTec AD — Bottle category — into Drive (only once, ever)
3. Build and test the `MVTecDataset` class
4. Visually verify image/mask alignment before writing any model code

**Do not skip step 4.** A silent bug here (misaligned mask, wrong normalization,
wrong train/test split) will silently corrupt every downstream result and you
won't find out until you're confused about why AUROC looks wrong on day 6.

## 1.1 — Mount Drive and check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.")

## 1.2 — Project paths (everything persistent lives under Drive)

Folder layout we will use for the **entire** project, not just this notebook:

```
/content/drive/MyDrive/anomaly_project/
├── data/mvtec/<category>/{train,test,ground_truth}/...
├── checkpoints/<category>/<strategy>_epoch<N>.pt
├── results/<category>_<strategy>.json
└── logs/
```

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/anomaly_project'
DATA_ROOT = f'{PROJECT_ROOT}/data/mvtec'
CHECKPOINT_ROOT = f'{PROJECT_ROOT}/checkpoints'
RESULTS_ROOT = f'{PROJECT_ROOT}/results'
LOG_ROOT = f'{PROJECT_ROOT}/logs'

for d in [DATA_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT, LOG_ROOT]:
    os.makedirs(d, exist_ok=True)

print("Project folders ready:")
for d in [PROJECT_ROOT, DATA_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT, LOG_ROOT]:
    print(f"  {d}")

## 1.3 — Extract MVTec AD (Bottle category) from the official tarball

**Why this approach:** Two earlier attempts to pull data through Hugging Face
mirrors (`datasets.load_dataset()`, then `fiftyone.utils.huggingface`) both
ran into either schema mismatches or unexplained network hangs during
download. Rather than keep fighting third-party mirrors, we now use the
**official MVTec source directly**: a single ~4.9GB tarball
(`mvtec_anomaly_detection.tar.xz`) downloaded once via the stable FTP link
MVTec itself provides, already saved to your Google Drive.

This is also better for your actual coursework -- citing the canonical
official dataset is more defensible than a third-party reformatted mirror.

**This notebook assumes the full tarball is already sitting in your Drive**
(per your manual download). The cell below extracts *only* the Bottle
category from it -- not all 15 categories -- so this stays fast even though
the source archive is large.

**Why no normalization is needed here:** unlike the Hugging Face mirrors
(which stored data as parquet/FiftyOne objects with non-obvious schemas), the
**official MVTec tarball already uses exactly the folder structure**
`MVTecDataset` expects:

```
<category>/
├── train/good/*.png
├── test/good/*.png
├── test/<defect_type>/*.png
└── ground_truth/<defect_type>/*_mask.png
```

So this cell is just a verification step, confirming extraction in 1.3
produced the expected layout before we build the dataset class on top of it.

In [ ]:
# 1.3b -- Verify extracted structure
#
# Unlike the earlier HF-mirror attempts, the OFFICIAL MVTec tarball already
# uses exactly the folder layout MVTecDataset expects:
#   <category>/train/good/*.png
#   <category>/test/<defect_type>/*.png   (and test/good/*.png)
#   <category>/ground_truth/<defect_type>/*_mask.png
# So there is no normalization/relabeling step needed here -- this cell only
# verifies that extraction in 1.3 produced the expected structure before we
# build the dataset class on top of it.

from pathlib import Path

def has_canonical_structure(path):
    p = Path(path)
    return (p / "train" / "good").exists() and (p / "ground_truth").exists()

assert has_canonical_structure(category_path), (
    f"Expected canonical structure at {category_path} but it's missing. "
    f"Re-run cell 1.3 (extraction) -- something went wrong."
)

print(f"Canonical structure confirmed for '{CATEGORY}':")
for sub in ["train/good", "test", "ground_truth"]:
    full = f"{category_path}/{sub}"
    if os.path.isdir(full):
        n_items = len(os.listdir(full))
        print(f"  {sub}/  -> {n_items} item(s)")

print("\nNo normalization needed -- official tarball structure matches")
print("MVTecDataset's expected layout exactly. Proceeding to section 1.4.")

## 1.4 — The `MVTecDataset` class

This is the single most important piece of code in the whole project. Every
downstream result — SSL pretraining, memory bank scoring, continual learning
forgetting curves — depends on this being correct. Two design choices to note:

- **`split='train'`** returns only normal (`good`) images, unlabeled — exactly
  what SimCLR/DINO pretraining needs.
- **`split='test'`** returns image + binary label (0=normal, 1=anomalous) +
  ground-truth mask (zeros for normal images) — exactly what PatchCore
  evaluation needs.

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from pathlib import Path


class MVTecDataset(Dataset):
    """
    MVTec AD dataset loader.

    Parameters
    ----------
    root : str
        Path to the category folder, e.g. '.../data/mvtec/bottle'
    split : str
        'train' -> normal images only, no labels (for SSL pretraining)
        'test'  -> all test images, with binary labels + masks (for eval)
    transform : callable, optional
        Applied to the PIL image. For 'train' with two_crop=True, the SAME
        transform is called twice independently (for contrastive pairs) --
        the transform itself must contain the randomness (RandomCrop etc).
    mask_transform : callable, optional
        Applied to the PIL mask. Must produce output spatially aligned with
        `transform`'s output -- i.e. same resize, but NO color jitter/flip
        unless the same flip is also applied to the image.
    two_crop : bool
        If True (used during SimCLR pretraining), __getitem__ returns a pair
        of independently-augmented views of the same image instead of one.
    """

    def __init__(self, root, split="train", transform=None, mask_transform=None, two_crop=False):
        self.root = Path(root)
        self.split = split
        self.transform = transform
        self.mask_transform = mask_transform
        self.two_crop = two_crop
        self.samples = self._build_index()

    def _build_index(self):
        samples = []
        if self.split == "train":
            good_dir = self.root / "train" / "good"
            if not good_dir.exists():
                raise FileNotFoundError(f"Expected {good_dir} to exist.")
            for img_path in sorted(good_dir.glob("*.png")):
                samples.append({"image_path": img_path, "label": 0, "mask_path": None})

        elif self.split == "test":
            test_dir = self.root / "test"
            if not test_dir.exists():
                raise FileNotFoundError(f"Expected {test_dir} to exist.")
            for defect_dir in sorted(test_dir.iterdir()):
                if not defect_dir.is_dir():
                    continue
                defect_type = defect_dir.name  # 'good' or e.g. 'broken_large'
                is_anomalous = defect_type != "good"
                for img_path in sorted(defect_dir.glob("*.png")):
                    if is_anomalous:
                        # MVTec convention: ground_truth/<defect_type>/<id>_mask.png
                        mask_path = (
                            self.root / "ground_truth" / defect_type / f"{img_path.stem}_mask.png"
                        )
                        if not mask_path.exists():
                            raise FileNotFoundError(
                                f"Missing ground truth mask for {img_path}. "
                                f"Expected at {mask_path}. Dataset structure may be corrupted."
                            )
                    else:
                        mask_path = None  # normal test images have no defect mask
                    samples.append(
                        {"image_path": img_path, "label": int(is_anomalous), "mask_path": mask_path}
                    )
        else:
            raise ValueError(f"split must be 'train' or 'test', got '{self.split}'")

        if len(samples) == 0:
            raise RuntimeError(f"No samples found for split='{self.split}' at {self.root}. "
                                f"Check that the dataset was downloaded/extracted correctly.")
        return samples

    def __len__(self):
        return len(self.samples)

    def _load_image(self, path):
        return Image.open(path).convert("RGB")

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = self._load_image(sample["image_path"])

        if self.split == "train":
            if self.two_crop:
                # Two independent stochastic augmentations of the SAME image --
                # this is the contrastive pair SimCLR's NT-Xent loss needs.
                view1 = self.transform(img) if self.transform else img
                view2 = self.transform(img) if self.transform else img
                return view1, view2
            else:
                view = self.transform(img) if self.transform else img
                return view

        else:  # split == 'test'
            img_t = self.transform(img) if self.transform else img

            if sample["mask_path"] is not None:
                mask = Image.open(sample["mask_path"]).convert("L")
                mask_t = self.mask_transform(mask) if self.mask_transform else mask
                mask_t = (np.array(mask_t) > 0).astype(np.float32)
                mask_t = torch.from_numpy(mask_t)
            else:
                # Normal image: mask is all-zero, same spatial size as the image
                # tensor so batching/stacking never breaks.
                if isinstance(img_t, torch.Tensor):
                    h, w = img_t.shape[-2:]
                else:
                    w, h = img.size
                mask_t = torch.zeros((h, w), dtype=torch.float32)

            return {
                "image": img_t,
                "label": sample["label"],
                "mask": mask_t,
                "image_path": str(sample["image_path"]),
            }

## 1.5 — Sanity checks (run these before trusting the dataset)

Three checks, each catching a different class of bug:

1. **Counts** — do train/test sizes roughly match what the MVTec paper reports for Bottle?
2. **Label balance** — test split should have both normal and anomalous images.
3. **Visual mask alignment** — does the anomaly mask actually sit on top of the defect?

In [ ]:
train_ds = MVTecDataset(category_path, split="train")
test_ds = MVTecDataset(category_path, split="test")

n_test_normal = sum(1 for s in test_ds.samples if s["label"] == 0)
n_test_anomalous = sum(1 for s in test_ds.samples if s["label"] == 1)

print(f"Category: {CATEGORY}")
print(f"  Train (normal only): {len(train_ds)} images")
print(f"  Test total: {len(test_ds)} images")
print(f"    -> normal:    {n_test_normal}")
print(f"    -> anomalous: {n_test_anomalous}")
print()
print("Reference (MVTec AD paper, Bottle category): 209 train / 83 test "
      "(20 good + 63 defective across 3 defect types)")

assert n_test_normal > 0, "BUG: no normal images in test split"
assert n_test_anomalous > 0, "BUG: no anomalous images in test split"
print("\nBasic count sanity checks PASSED.")

In [ ]:
# Visual mask alignment check -- THE most important cell in this notebook.
# If the mask doesn't overlay the defect correctly here, stop and debug
# before writing any more code.

import matplotlib.pyplot as plt

anomalous_indices = [i for i, s in enumerate(test_ds.samples) if s["label"] == 1]
sample_indices = anomalous_indices[:3]  # first 3 defective examples

fig, axes = plt.subplots(len(sample_indices), 3, figsize=(12, 4 * len(sample_indices)))

for row, idx in enumerate(sample_indices):
    sample = test_ds[idx]
    img = sample["image"]
    mask = sample["mask"].numpy()
    defect_type = Path(sample["image_path"]).parent.name

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"Image ({defect_type})")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title("Ground truth mask")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(img)
    axes[row, 2].imshow(mask, cmap="Reds", alpha=0.4)
    axes[row, 2].set_title("Overlay (mask should sit ON the defect)")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

print("\nCHECK: In the rightmost column, the red highlight should sit exactly")
print("on top of the visible defect (crack, contamination, etc). If it doesn't,")
print("do not proceed -- there is a path-matching or resizing bug to fix first.")

In [ ]:
# Visual check on the TRAIN split: confirm these are genuinely defect-free
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i in range(4):
    img = train_ds[i]
    axes[i].imshow(img)
    axes[i].set_title(f"train[{i}] (should be defect-free)")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## 1.6 — Checkpointing utility (used from Notebook 2 onward)

Building this now, before any training code exists, so the habit is established
from the very first training loop rather than retrofitted after a disconnect
costs you a wasted GPU session.

In [ ]:
import json
import time


def save_checkpoint(state: dict, category: str, strategy: str, tag: str = "latest"):
    """
    Save a training checkpoint to Drive.

    state must be a dict of plain tensors/python objects (e.g.
    {'model_state': model.state_dict(), 'epoch': epoch, 'optimizer_state': ...}).
    Saved to CHECKPOINT_ROOT/<category>/<strategy>_<tag>.pt
    """
    out_dir = f"{CHECKPOINT_ROOT}/{category}"
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{strategy}_{tag}.pt"
    state["_saved_at"] = time.time()
    torch.save(state, path)
    print(f"Checkpoint saved: {path}")
    return path


def load_checkpoint(category: str, strategy: str, tag: str = "latest", map_location="cpu"):
    path = f"{CHECKPOINT_ROOT}/{category}/{strategy}_{tag}.pt"
    if not os.path.exists(path):
        print(f"No checkpoint found at {path} -- starting fresh.")
        return None
    state = torch.load(path, map_location=map_location)
    age_min = (time.time() - state.get("_saved_at", time.time())) / 60
    print(f"Loaded checkpoint from {path} (saved {age_min:.1f} min ago)")
    return state


def save_result(result: dict, category: str, strategy: str):
    """Append/overwrite a result record under RESULTS_ROOT/<category>_<strategy>.json"""
    path = f"{RESULTS_ROOT}/{category}_{strategy}.json"
    with open(path, "w") as f:
        json.dump(result, f, indent=2, default=str)
    print(f"Result saved: {path}")


# Smoke test -- not a real checkpoint, just confirms write/read works end to end
_test_state = {"dummy": torch.tensor([1, 2, 3]), "epoch": 0}
_p = save_checkpoint(_test_state, category=CATEGORY, strategy="smoketest")
_loaded = load_checkpoint(category=CATEGORY, strategy="smoketest")
assert torch.equal(_loaded["dummy"], _test_state["dummy"]), "Checkpoint roundtrip failed!"
os.remove(_p)
print("\nCheckpoint roundtrip PASSED. Drive persistence is working correctly.")

## Step 1 — Done. What you should have right now:

- [ ] Drive mounted, GPU confirmed
- [ ] Bottle category downloaded into `DATA_ROOT` on Drive (one-time)
- [ ] `MVTecDataset` class working for both `train` and `test` splits
- [ ] Visual confirmation that masks align with defects (the most important check)
- [ ] Checkpoint save/load utility verified end-to-end

**Next step (Notebook 2):** SimCLR pretraining on the `train` split using
`MVTecDataset(..., two_crop=True)`, with augmentations tuned for industrial
imagery (reduced color jitter -- see proposal Section 3.3).

**Before moving on:** re-run this same notebook with `CATEGORY = "cable"` and
`CATEGORY = "leather"` to confirm the pipeline generalizes across categories
without code changes -- only the folder structure should differ, and the
dataset class should handle that transparently.